In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    """多头注意力机制"""
    def __init__(self, d_model, nhead, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.nhead = nhead
        self.d_k = d_model // nhead
        
        self.q_linear = nn.Linear(d_model, d_model)
        self.k_linear = nn.Linear(d_model, d_model)
        self.v_linear = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
        self.out_linear = nn.Linear(d_model, d_model)

    def forward(self, q, k, v, mask=None):
        batch_size = q.size(0)
        
        # 线性变换并分头
        q = self.q_linear(q).view(batch_size, -1, self.nhead, self.d_k).transpose(1,2)
        k = self.k_linear(k).view(batch_size, -1, self.nhead, self.d_k).transpose(1,2)
        v = self.v_linear(v).view(batch_size, -1, self.nhead, self.d_k).transpose(1,2)
        
        # 计算注意力
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        
        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        
        # 合并多头
        output = torch.matmul(attn, v)
        output = output.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        return self.out_linear(output)

class FeedForward(nn.Module):
    """前馈网络"""
    def __init__(self, d_model, dim_feedforward=2048, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        
    def forward(self, x):
        x = F.relu(self.linear1(x))
        x = self.dropout(x)
        return self.linear2(x)

class EncoderLayer(nn.Module):
    """编码器层"""
    def __init__(self, d_model, nhead, dim_feedforward, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, nhead, dropout)
        self.ffn = FeedForward(d_model, dim_feedforward, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        
    def forward(self, src, src_mask=None):
        # 自注意力
        src2 = self.self_attn(src, src, src, src_mask)
        src = src + self.dropout1(src2)
        src = self.norm1(src)
        
        # 前馈网络
        src2 = self.ffn(src)
        src = src + self.dropout2(src2)
        return self.norm2(src)

class DecoderLayer(nn.Module):
    """解码器层"""
    def __init__(self, d_model, nhead, dim_feedforward, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, nhead, dropout)
        self.cross_attn = MultiHeadAttention(d_model, nhead, dropout)
        self.ffn = FeedForward(d_model, dim_feedforward, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)
        
    def forward(self, tgt, memory, tgt_mask=None, memory_mask=None):
        # 自注意力（带掩码）
        tgt2 = self.self_attn(tgt, tgt, tgt, tgt_mask)
        tgt = tgt + self.dropout1(tgt2)
        tgt = self.norm1(tgt)
        
        # 交叉注意力（编码器输出作为k,v）
        tgt2 = self.cross_attn(tgt, memory, memory, memory_mask)
        tgt = tgt + self.dropout2(tgt2)
        tgt = self.norm2(tgt)
        
        # 前馈网络
        tgt2 = self.ffn(tgt)
        tgt = tgt + self.dropout3(tgt2)
        return self.norm3(tgt)

class TransformerEncoder(nn.Module):
    """完整编码器"""
    def __init__(self, num_layers, d_model, nhead, dim_feedforward, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            EncoderLayer(d_model, nhead, dim_feedforward, dropout)
            for _ in range(num_layers)
        ])
        
    def forward(self, src, src_mask=None):
        for layer in self.layers:
            src = layer(src, src_mask)
        return src

class TransformerDecoder(nn.Module):
    """完整解码器"""
    def __init__(self, num_layers, d_model, nhead, dim_feedforward, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            DecoderLayer(d_model, nhead, dim_feedforward, dropout)
            for _ in range(num_layers)
        ])
        
    def forward(self, tgt, memory, tgt_mask=None, memory_mask=None):
        for layer in self.layers:
            tgt = layer(tgt, memory, tgt_mask, memory_mask)
        return tgt

In [2]:
import torch
import torch.nn as nn
import json
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer
import math
import os

class QADataset(Dataset):
    """SQuAD格式数据处理"""
    def __init__(self, file_path, tokenizer, max_length=256):
        self.data = []
        with open(file_path) as f:
            squad_data = json.load(f)
        flag = 1
        for article in squad_data['data'][:200]:
            for paragraph in article['paragraphs']:
                context = paragraph['context']
                for qa in paragraph['qas']:
                    question = qa['question']
                    answer = qa['answers'][0]['text'] if qa['answers'] else ""
                    
                    # 构造模型输入输出
                    input_text = f"{question} context: {context}"
                    output_text = f"{answer}"
                    
                    if flag:
                        print("input_text:", input_text)
                        print("output text:", output_text)
                        flag = 0

                    # 编码文本
                    inputs = tokenizer(input_text, max_length=max_length, 
                                     padding='max_length', truncation=True, 
                                     return_tensors="pt")
                    
                    targets = tokenizer(output_text, max_length=max_length,
                                      padding='max_length', truncation=True,
                                      return_tensors="pt")
                    
                    self.data.append({
                        'input_ids': inputs['input_ids'].squeeze(0),
                        'attention_mask': inputs['attention_mask'].squeeze(0),
                        'labels': targets['input_ids'].squeeze(0)
                    })
        # self.data = self.data[:1]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]


class PositionalEncoding(nn.Module):
    """位置编码模块"""
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:x.size(1), :]
        return self.dropout(x)

class TransformerQA(nn.Module):
    """问答模型（含自定义编码器/解码器）"""
    def __init__(self, vocab_size, d_model=512, nhead=8, 
                 num_encoder_layers=6, num_decoder_layers=6,
                 dim_feedforward=2048, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model, dropout=dropout)
        
        self.encoder = TransformerEncoder(
            num_encoder_layers, d_model, nhead, dim_feedforward, dropout)
        
        self.decoder = TransformerDecoder(
            num_decoder_layers, d_model, nhead, dim_feedforward, dropout)
        
        self.fc_out = nn.Linear(d_model, vocab_size)
        self.d_model = d_model

    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        # 编码器处理
        src = self.embedding(src) * math.sqrt(self.d_model)
        src = self.pos_encoder(src)
        memory = self.encoder(src, src_mask)
        
        # 解码器处理
        tgt = self.embedding(tgt) * math.sqrt(self.d_model)
        tgt = self.pos_encoder(tgt)
        output = self.decoder(tgt, memory, tgt_mask)
        
        return self.fc_out(output)

    def generate_mask(self, src, tgt):
        """生成注意力掩码"""
        # 源序列填充掩码
        src_pad_mask = (src == 0).unsqueeze(1).unsqueeze(2)
        
        # 目标序列掩码（自回归+填充）
        tgt_pad_mask = (tgt == 0).unsqueeze(1).unsqueeze(2)
        seq_len = tgt.size(1)
        tgt_sub_mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
        tgt_mask = tgt_pad_mask | tgt_sub_mask.to(src.device)
        
        return src_pad_mask, tgt_mask
        
    
    def generate_answer(self, src, tokenizer, max_len=50):
        src = self.embedding(src) * math.sqrt(self.d_model)
        src = self.pos_encoder(src)
        memory = self.encoder(src)

        outputs = torch.LongTensor([[tokenizer.cls_token_id]]).to(src.device)

        for _ in range(max_len):
            tgt = self.embedding(outputs) * math.sqrt(self.d_model)
            tgt = self.pos_encoder(tgt)

            out = self.decoder(tgt, memory)
            
            next_token = out.argmax(-1)[:, -1:]
           
            outputs = torch.cat([outputs, next_token], dim=-1)

            if next_token.item() == tokenizer.sep_token_id:
                break

        return tokenizer.decode(outputs.squeeze(0), skip_special_tokens=True)


class QATrainer:
    """训练管理类"""
    def __init__(self, model, tokenizer, device='cuda'):
        self.model = model.to(device)
        self.tokenizer = tokenizer
        self.device = device
        self.criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)
        
    def prepare_batch(self, batch):
        """数据预处理"""
        src = batch['input_ids'].to(self.device)   # 源序列（问题+上下文）
        tgt = batch['labels'].to(self.device)      # 目标序列（答案）
        # 注意：batch是否在前。
        
        # 构造解码器输入（去尾）和输出（去头）
        decoder_input = tgt[:, :-1]  # 移除最后一个token
        decoder_output = tgt[:, 1:]  # 移除第一个token
        
        # 生成注意力掩码
        src_mask, tgt_mask = self.model.generate_mask(src, decoder_input)
        return src, decoder_input, decoder_output, src_mask, tgt_mask
        
    def train_epoch(self, dataloader, optimizer):
        self.model.train()
        total_loss = 0
        
        # print(len(dataloader))
        # exit()
        #progress_bar = tqdm(enumerate(dataloader), total=len(dataloader), leave=False)
        for batch_idx, batch in enumerate(dataloader):
            # 数据准备
            src, decoder_in, decoder_out, src_mask, tgt_mask = self.prepare_batch(batch)

            optimizer.zero_grad()
            
            # 前向传播 (B, S) -> (B, S, V)
            # 直接使用目标序列来训练
            outputs = self.model(
                src,#.transpose(0, 1),   # (S, B) 时间维在前
                decoder_in,#.transpose(0, 1),
                src_mask,
                tgt_mask
            )#.transpose(0, 1)  # 转回(B, S, V)
            
            # print(torch.argmax(outputs, dim=-1), decoder_out)

            # 计算损失
            loss = self.criterion(
                outputs.reshape(-1, outputs.shape[-1]), 
                decoder_out.reshape(-1)
            )
            
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            
            #progress_bar.set_description(f"Loss: {loss.item():.4f}")
            #if batch_idx % 100 == 0:
                #current_loss = total_loss / (batch_idx + 1)
                #print(f"  Batch {batch_idx} | Loss: {current_loss:.4f}")
                #current_loss = total_loss / (batch_idx + 1)
                #progress_bar.set_postfix(avg_loss=f"{current_loss:.4f}")
                
            
        return total_loss / len(dataloader)

    def evaluate(self, dataloader):
        self.model.eval()
        total_loss = 0

        with torch.no_grad():
            for batch_idx, batch in enumerate(dataloader):
                src, decoder_in, decoder_out, src_mask, tgt_mask = self.prepare_batch(batch)
                
                outputs = self.model(
                    src,
                    decoder_in,
                    src_mask,
                    tgt_mask
                )
                
                loss = self.criterion(
                    outputs.reshape(-1, outputs.shape[-1]),
                    decoder_out.reshape(-1)
                )
                
                total_loss += loss.item()

        return total_loss / len(dataloader)

In [3]:
from tqdm import tqdm
import torch

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
train_dataset = QADataset('/kaggle/input/squad-v2-0/SQuAD-train-v2.0.json', tokenizer)
#train_dataset = train_dataset[:100000]
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
model = TransformerQA(vocab_size=tokenizer.vocab_size)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
trainer = QATrainer(model, tokenizer, device)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-5)
num_epochs = 10

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch + 1}/{num_epochs}")
    
    # 使用tqdm包装train_loader
    train_loader_tqdm = tqdm(train_loader, desc=f"Training Epoch {epoch + 1}", leave=True)
    
    train_loss = trainer.train_epoch(train_loader_tqdm, optimizer)
    
    print(f"\nTrain Loss: {train_loss:.4f}")
    model_save_path = f'/kaggle/working/model_epoch_{epoch + 1}.pth'
    torch.save(trainer.model.state_dict(), model_save_path)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

input_text: When did Beyonce start becoming popular? context: Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ bee-YON-say) (born September 4, 1981) is an American singer, songwriter, record producer and actress. Born and raised in Houston, Texas, she performed in various singing and dancing competitions as a child, and rose to fame in the late 1990s as lead singer of R&B girl-group Destiny's Child. Managed by her father, Mathew Knowles, the group became one of the world's best-selling girl groups of all time. Their hiatus saw the release of Beyoncé's debut album, Dangerously in Love (2003), which established her as a solo artist worldwide, earned five Grammy Awards and featured the Billboard Hot 100 number-one singles "Crazy in Love" and "Baby Boy".
output text: in the late 1990s

Epoch 1/10


...


Train Loss: 0.3417

Epoch 9/10


Training Epoch 9: 100%|██████████| 3662/3662 [32:39<00:00,  1.87it/s]



Train Loss: 0.2359

Epoch 10/10


Training Epoch 10: 100%|██████████| 3662/3662 [32:39<00:00,  1.87it/s]



Train Loss: 0.1639
